# Kerala Essential Commodities: Price & Supply-Stress Intelligence
### Run the full pipeline step by step

This notebook runs the same logic as `scripts/` but lets you look at every intermediate table, change the scoring settings and redraw the charts.

**How to run:** install the requirements (`pip install -r requirements.txt`), start Jupyter from the repo root (`jupyter lab`), open this notebook and choose **Run > Run All Cells**. It also works if you start Jupyter from inside the `notebooks/` folder.

| Step | What happens | Output |
|---|---|---|
| 1 | Load and check the raw daily prices | tables |
| 2 | Aggregate daily to monthly (state average, dispersion, breadth, district premium) | `build/*.pkl` |
| 3 | Cross-check against the archived monthly file | difference stats |
| 4 | Score risk (Red / Amber / Green) | alert tables |
| 5 | Test other weights (sensitivity) | top 10 comparison |
| 6 | Charts | coconut oil, alerts per month, group heatmap, district premium |
| 7 | Build the Excel workbook | `build/Kerala_Commodity_Price_Risk_MIS_rebuilt.xlsx` |
| 8 | Check Excel formulas against the pandas scores | max difference |

## 0. Setup

In [ ]:
from pathlib import Path
import os, sys, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 100)

# Find the repo: look for data/fact_daily_prices_clean.csv in this folder, its parents, then one level down
_marker = Path('data') / 'fact_daily_prices_clean.csv'
_cands = [p for p in [Path.cwd(), *Path.cwd().parents] if (p / _marker).exists()]
if not _cands:
    _cands = [p.parents[1] for p in Path.cwd().glob('*/' + str(_marker))]
assert _cands, 'Could not find the repo (data/fact_daily_prices_clean.csv). Open this notebook from inside the repo folder.'
ROOT = _cands[0]
DATA, BUILD = ROOT / 'data', ROOT / 'build'
BUILD.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'scripts'))   # makes monthly_aggregation.py and risk_score.py importable
print('Repo root:', ROOT)

## 1. Load and check the raw data
The fact table has one row per date, district and commodity price. Dimension tables hold names, groups and units.

In [ ]:
fact = pd.read_csv(DATA / 'fact_daily_prices_clean.csv')
items = pd.read_csv(DATA / 'dim_item.csv')
districts = pd.read_csv(DATA / 'dim_district.csv')
dates = pd.read_csv(DATA / 'dim_date.csv', parse_dates=['full_date'])
fact = fact.merge(dates[['date_id', 'full_date', 'year']], on='date_id')
print(f"{len(fact):,} rows | {fact.full_date.min():%d %b %Y} to {fact.full_date.max():%d %b %Y} | "
      f"{fact.district_id.nunique()} districts | {fact.item_id.nunique()} commodities")
fact.head()

In [ ]:
checks = {
    'missing prices': int(fact.price.isna().sum()),
    'non-positive prices': int((fact.price <= 0).sum()),
    'duplicate (date, district, item) rows': int(fact.duplicated(['date_id', 'district_id', 'item_id']).sum()),
}
print(checks)

n_days = fact.full_date.nunique()
print(f'Overall cell coverage: {len(fact) / (len(districts) * len(items) * n_days):.1%} '
      f'(rows / districts x commodities x reporting days)')

coverage = (fact.groupby('item_id').size() / (len(districts) * n_days)).rename('coverage').reset_index()
coverage = coverage.merge(items[['item_id', 'item_name_clean', 'unit']], on='item_id')
coverage.sort_values('coverage').head(10)   # thinnest commodities

## 2. Aggregate daily prices to monthly metrics
`scripts/monthly_aggregation.py` does the heavy lifting:

* **District-month mean:** needs at least 5 daily observations to count.
* **State average:** equal-weight mean of the district monthly means.
* **District dispersion:** coefficient of variation across districts (needs 3+ districts).
* **Breadth of rise:** share of districts more than 10% above their own prior-12-month average.
* **District premium:** district mean / state average - 1.

In [ ]:
from monthly_aggregation import aggregate, save

res = aggregate(DATA)
m, dpm, md_, sea, di, dd = (res[k] for k in ['m', 'dpm', 'md', 'sea', 'di', 'dd'])
print('commodity x month rows:', len(m), '| district x commodity x month rows:', len(dpm))
m.head(8)

In [ ]:
# Look at one commodity
tomato = m.merge(di[['item_id', 'label']], on='item_id').query("label == 'Tomato [Kg]'")
tomato[['month', 'state_avg', 'n_districts', 'disp_cv', 'breadth']].tail(8).round(3)

In [ ]:
save(res, BUILD)   # writes build/*.pkl (needed by scripts/03_build_workbook.py)
sorted(p.name for p in BUILD.glob('*.pkl'))

## 3. Cross-check against the archived monthly file
`archive/monthly_price_changes.csv` was produced earlier with a slightly different averaging (all daily rows pooled). The two should agree closely.

In [ ]:
archive = pd.read_csv(ROOT / 'archive' / 'monthly_price_changes.csv')
archive['month'] = pd.to_datetime(archive.year_month + '-01')
cmp = archive.merge(m, on=['item_id', 'month'])
cmp['abs_diff'] = (cmp.avg_price / cmp.state_avg - 1).abs()
print(cmp.abs_diff.describe(percentiles=[.5, .95]).round(4))
cmp.sort_values('abs_diff', ascending=False).head(5)[['item_name_clean', 'month', 'avg_price', 'state_avg', 'n_districts', 'abs_diff']]

## 4. Risk scoring
`scripts/risk_score.py` holds the weights (`W`), caps (`CAP`) and thresholds (`TH`). They mirror the **Assumptions** sheet of the workbook. Each component is `MIN(1, value / cap) x 100`; the risk score is the weighted average; Red >= 65, Amber >= 40.

In [ ]:
from risk_score import score, W, CAP, TH
print('weights   ', W)
print('caps      ', CAP)
print('thresholds', TH)

scored = score(m, sea).merge(di[['item_id', 'label', 'item_group']], on='item_id')
elig = scored[scored.elig == 1]     # scored = 12+ months of history and at least 5 districts reporting
print(f'{len(elig):,} scored commodity-months, {elig.month.min():%b %Y} to {elig.month.max():%b %Y}')
elig.groupby('month').cls.value_counts().unstack(fill_value=0).tail(12)

In [ ]:
MONTH = pd.Timestamp('2026-02-01')    # <-- change me: any month from 2024-01-01 to 2026-02-01
cols = ['label', 'state_avg', 'mom', 'yoy', 'dev', 'vol', 'disp_cv', 'breadth', 'risk', 'cls', 'type', 'signal', 'n_districts']
top = elig[elig.month == MONTH].sort_values('risk', ascending=False)[cols].head(10)
top.round(3)

## 5. Sensitivity: what if the weights were different?
The weights are judgement calls, so test them. Edit `alt_w` (any numbers; they are re-scaled to sum to 100%) and see how far the top 10 moves.

In [ ]:
def top_labels(weights=W, caps=CAP, thresholds=TH, month=MONTH, n=10):
    s = score(m, sea, W=weights, CAP=caps, TH=thresholds).merge(di[['item_id', 'label']], on='item_id')
    s = s[(s.elig == 1) & (s.month == month)].sort_values('risk', ascending=False)
    return s[['label', 'risk', 'cls']].head(n).reset_index(drop=True)

base = top_labels()
alt_w = {**W, 'sea': 0.0, 'mom': 0.30}      # example: drop seasonality, lean on momentum
alt = top_labels(weights=alt_w)
overlap = len(set(base.label) & set(alt.label))
print(f'{overlap} of the top 10 are the same under the alternative weights')
pd.concat([base.add_suffix('_base'), alt.add_suffix('_alt')], axis=1).round(1)

## 6. Charts

In [ ]:
co = scored[scored.label == 'Coconut oil( Kera) [Ltr]'].sort_values('month')
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(co.month, co.state_avg, label='State average price (Rs/L)')
ax[0].plot(co.month, co.avg12, label='Prior 12M average', ls='--')
ax[0].legend(); ax[0].set_title('Coconut oil (Kera): a steady surge, low volatility')
ax[1].plot(co.month, co.risk, color='firebrick')
ax[1].axhline(TH['red'], ls=':', c='grey'); ax[1].axhline(TH['amber'], ls=':', c='grey')
ax[1].set_ylabel('Risk score'); ax[1].set_ylim(0, 100)
plt.tight_layout(); plt.show()

In [ ]:
counts = elig.groupby(['month', 'cls']).size().unstack(fill_value=0)
x = np.arange(len(counts))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x, counts['Red'], color='firebrick', label='Red')
ax.bar(x, counts['Amber'], bottom=counts['Red'], color='orange', label='Amber')
ax.set_xticks(x); ax.set_xticklabels([d.strftime('%b %y') for d in counts.index], rotation=90)
ax.set_title('Red and Amber alerts per month'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
scored['grp'] = scored.item_group.map(lambda g: ' '.join(str(g).split()))
grp = scored[scored.elig == 1].groupby(['grp', 'month']).risk.mean().unstack()
last12 = grp.iloc[:, -12:]
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(last12.values, aspect='auto', cmap='RdYlGn_r', vmin=10, vmax=70)
ax.set_yticks(range(len(last12))); ax.set_yticklabels(last12.index)
ax.set_xticks(range(12)); ax.set_xticklabels([d.strftime('%b %y') for d in last12.columns], rotation=45)
plt.colorbar(im, label='Average risk score'); ax.set_title('Average risk score by group, last 12 months')
plt.tight_layout(); plt.show()

In [ ]:
sel = dpm[(dpm.month == MONTH) & (dpm.eligible == 1)]
dist = sel.groupby('district_name').premium.mean().sort_values(ascending=False)
ax = dist.mul(100).plot(kind='barh', figsize=(7, 4.5), color=['firebrick' if v > 0 else 'seagreen' for v in dist])
ax.invert_yaxis(); ax.axvline(0, c='k', lw=0.8)
ax.set_xlabel('Average price premium vs state average (%)'); ax.set_title(f'District price premium, {MONTH:%Y-%m}')
plt.tight_layout(); plt.show()

## 7. Build the Excel workbook
This runs `scripts/03_build_workbook.py`. The output goes to `build/` so the shipped, already-recalculated workbook in the repo root is **not** overwritten.

The rebuilt file contains formulas without cached values: open it in Excel (it calculates on load) or recalculate once with LibreOffice before previewing it elsewhere.

In [ ]:
import runpy
os.environ['MIS_OUT'] = str(BUILD / 'Kerala_Commodity_Price_Risk_MIS_rebuilt.xlsx')
runpy.run_path(str(ROOT / 'scripts' / '03_build_workbook.py'), run_name='__main__')

## 8. Check the Excel formulas against the pandas scores
Compares every scored commodity-month. It checks the rebuilt workbook if that file already has calculated values (open it in Excel and save it once), otherwise it falls back to the shipped, already-recalculated workbook and says so.

In [ ]:
from openpyxl import load_workbook

def load_calc(path):
    ws = load_workbook(path, data_only=True)['Data_Calc']
    hdr = [c.value for c in ws[1]]
    return pd.DataFrame(list(ws.iter_rows(min_row=2, values_only=True)), columns=hdr)

REBUILT = BUILD / 'Kerala_Commodity_Price_Risk_MIS_rebuilt.xlsx'
SHIPPED = ROOT / 'Kerala_Commodity_Price_Risk_MIS.xlsx'
xl, used = (load_calc(REBUILT), REBUILT.name) if REBUILT.exists() else (None, None)
if xl is None or pd.to_numeric(xl['RISK SCORE'], errors='coerce').isna().all():
    print('The rebuilt workbook has no cached values yet (open it in Excel and save it to include it here).')
    xl, used = load_calc(SHIPPED), SHIPPED.name
xl['month'] = pd.to_datetime(xl['Month'])
joined = scored.merge(xl, on=['item_id', 'month'])
e = joined[joined.elig == 1]
risk_xl = pd.to_numeric(e['RISK SCORE'], errors='coerce')
print(f"Checked: {used}")
print(f"{len(e):,} scored rows | max risk-score difference {(risk_xl - e.risk).abs().max():.2e} | "
      f"class mismatches {(e['Class'] != e.cls).sum()} | risk-type mismatches {(e['Risk type'] != e.type).sum()}")